<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [19]:
from functools import lru_cache

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split


@lru_cache(maxsize=1)
def _fetch_openml_dataset():
    """Baixa o dataset uma vez e reutiliza em chamadas seguintes."""
    candidates = [
        ("Fashion-MNIST", {"version": 1}),
        ("mnist_784", {}),
    ]

    last_error = None
    for name, extra_kwargs in candidates:
        try:
            X, y = fetch_openml(
                name=name,
                as_frame=False,
                parser="auto",
                return_X_y=True,
                **extra_kwargs,
            )
            return X.astype(np.float32), y.astype(np.int64), name
        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Nao foi possivel carregar o dataset via OpenML. "
        f"Ultimo erro: {last_error}"
    )


def load_data(seed=42, test_size=0.2, max_samples=15000, normalize=False):
    """Carrega dados, aplica amostragem opcional e divide em treino/teste."""
    X, y, _ = _fetch_openml_dataset()

    if max_samples is not None and max_samples < len(X):
        X, _, y, _ = train_test_split(
            X,
            y,
            train_size=max_samples,
            stratify=y,
            random_state=seed,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )

    if normalize:
        X_train = X_train / 255.0
        X_test = X_test / 255.0

    return X_train, X_test, y_train, y_test

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [20]:
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier


def train_random_forest(
    X_train,
    y_train,
    seed=42,
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    **kwargs,
):
    """Treina Random Forest com controle total de aleatoriedade."""
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=seed,
        n_jobs=n_jobs,
        **kwargs,
    )
    model.fit(X_train, y_train)
    return model


def train_adaboost(
    X_train,
    y_train,
    seed=42,
    n_estimators=80,
    learning_rate=0.7,
    base_depth=2,
    **kwargs,
):
    """Treina AdaBoost com estimador base raso para ganho de desempenho."""
    base_estimator = DecisionTreeClassifier(max_depth=base_depth, random_state=seed)

    # Compatibilidade com diferentes versoes do scikit-learn.
    model_kwargs = dict(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=seed,
    )
    model_kwargs.update(kwargs)

    if "estimator" in AdaBoostClassifier().get_params():
        model_kwargs["estimator"] = base_estimator
    else:
        model_kwargs["base_estimator"] = base_estimator

    model = AdaBoostClassifier(**model_kwargs)
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [21]:
def evaluate(model, X_test, y_test, average="macro", return_metrics=False):
    """Avalia o modelo com multiplas metricas e retorno flexivel."""
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average=average, zero_division=0
    )

    metrics = {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }
    return metrics if return_metrics else metrics["accuracy"]

**Resposta (normalização):**
Para Random Forest e AdaBoost com árvores como estimadores base, a normalização não é obrigatória porque os splits das árvores dependem da ordem dos valores, não da escala absoluta.
Ainda assim, manter a opção de normalizar no pipeline é útil para comparabilidade e para reaproveitar a estrutura caso outros modelos sensíveis à escala sejam adicionados no futuro (ex.: regressão logística, SVM, k-NN).

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [22]:
def run_pipeline(
    model_type="rf",
    seed=42,
    normalize=False,
    max_samples=15000,
    return_metrics=False,
    **model_kwargs,
):
    """Executa o pipeline completo para RF ou AdaBoost."""
    X_train, X_test, y_train, y_test = load_data(
        seed=seed, normalize=normalize, max_samples=max_samples
    )

    model_key = model_type.lower()
    if model_key == "rf":
        model = train_random_forest(X_train, y_train, seed=seed, **model_kwargs)
    elif model_key == "ab":
        model = train_adaboost(X_train, y_train, seed=seed, **model_kwargs)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'.")

    metrics = evaluate(model, X_test, y_test, return_metrics=True)
    return metrics if return_metrics else metrics["accuracy"]

**Existe overfitting?**
A resposta vem da função `overfitting_report`: quando `train_accuracy` é muito maior que `test_accuracy`, há overfitting.

**Qual modelo tende a sofrer mais?**
Em geral, Random Forest com árvores profundas tende a memorizar melhor o treino e pode ter maior gap de generalização se o número de árvores/profundidade crescer sem controle.

**Observação:**
Diferente da árvore única com `max_depth=None`, Random Forest agrega várias árvores e reduz variância média, mas ainda pode superajustar dependendo do conjunto de hiperparâmetros e do tamanho amostral.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [23]:
import pandas as pd


def compare_models(seed=42, max_samples=15000):
    """Compara RF e AdaBoost com as metricas solicitadas."""
    rows = []
    for model_type in ["rf", "ab"]:
        metrics = run_pipeline(
            model_type=model_type,
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
        )
        metrics["model"] = "Random Forest" if model_type == "rf" else "AdaBoost"
        rows.append(metrics)

    result = pd.DataFrame(rows)[["model", "accuracy", "precision", "recall", "f1"]]
    return result.sort_values(by="f1", ascending=False).reset_index(drop=True)


def best_initial_model(seed=42, max_samples=15000):
    table = compare_models(seed=seed, max_samples=max_samples)
    return table.iloc[0]["model"], table

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [24]:
def compare_seeds(seeds=(42, 7), model_types=("rf", "ab"), max_samples=15000):
    """Avalia estabilidade do experimento para diferentes sementes."""
    rows = []
    for seed in seeds:
        for model_type in model_types:
            metrics = run_pipeline(
                model_type=model_type,
                seed=seed,
                max_samples=max_samples,
                return_metrics=True,
            )
            rows.append(
                {
                    "seed": seed,
                    "model": model_type,
                    "accuracy": metrics["accuracy"],
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                }
            )
    return pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)


def reproducibility_check(model_type="rf", seed=42):
    acc1 = run_pipeline(model_type=model_type, seed=seed)
    acc2 = run_pipeline(model_type=model_type, seed=seed)
    return {
        "first_run": acc1,
        "second_run": acc2,
        "absolute_diff": abs(acc1 - acc2),
    }

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [25]:
def overfitting_report(seed=42, max_samples=15000):
    """Compara treino vs teste para identificar overfitting nos modelos."""
    X_train, X_test, y_train, y_test = load_data(seed=seed, max_samples=max_samples)

    rf = train_random_forest(X_train, y_train, seed=seed, n_estimators=250)
    ab = train_adaboost(X_train, y_train, seed=seed, n_estimators=120)

    rows = []
    for name, model in [("Random Forest", rf), ("AdaBoost", ab)]:
        train_acc = float(model.score(X_train, y_train))
        test_acc = float(model.score(X_test, y_test))
        rows.append(
            {
                "model": name,
                "train_accuracy": train_acc,
                "test_accuracy": test_acc,
                "generalization_gap": train_acc - test_acc,
            }
        )
    return pd.DataFrame(rows).sort_values("generalization_gap", ascending=False)

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [26]:
def hyperparameter_sensitivity(seed=42, max_samples=15000):
    """Varia n_estimators em RF e AdaBoost e compara impacto no desempenho."""
    rf_estimators = [50, 100, 200]
    ab_estimators = [30, 60, 120]

    rows = []
    for n in rf_estimators:
        metrics = run_pipeline(
            model_type="rf",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "rf", "n_estimators": n, **metrics})

    for n in ab_estimators:
        metrics = run_pipeline(
            model_type="ab",
            seed=seed,
            max_samples=max_samples,
            return_metrics=True,
            n_estimators=n,
        )
        rows.append({"model": "ab", "n_estimators": n, **metrics})

    return pd.DataFrame(rows).sort_values(["model", "n_estimators"]).reset_index(drop=True)

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [18]:
critical_reflection = {
    "1_acuracia_suficiente": (
        "Nao. Acuracia sozinha pode esconder erros relevantes em classes especificas; "
        "por isso o pipeline calcula tambem precisao, recall e F1 macro."
    ),
    "2_nao_por_acaso": (
        "O experimento fixa random_state em todas as etapas e repete execucoes com a mesma seed, "
        "verificando estabilidade numerica. Tambem compara seeds diferentes para medir variabilidade."
    ),
    "3_problemas_metodologicos": [
        "(i) Dependencia de uma unica divisao treino/teste pode enviesar conclusoes; validacao cruzada ajudaria.",
        "(ii) Testar muitos hiperparametros sem protocolo fixo pode induzir overfitting no conjunto de teste.",
    ],
    "4_pipeline_confiavel": (
        "Parcialmente confiavel: ele e reprodutivel, modular e mede multiplas metricas, "
        "mas ainda precisa de validacao cruzada e analise de erro por classe para maior robustez."
    ),
}

critical_reflection

{'1_acuracia_suficiente': 'Nao. Acuracia sozinha pode esconder erros relevantes em classes especificas; por isso o pipeline calcula tambem precisao, recall e F1 macro.',
 '2_nao_por_acaso': 'O experimento fixa random_state em todas as etapas e repete execucoes com a mesma seed, verificando estabilidade numerica. Tambem compara seeds diferentes para medir variabilidade.',
 '3_problemas_metodologicos': ['(i) Dependencia de uma unica divisao treino/teste pode enviesar conclusoes; validacao cruzada ajudaria.',
  '(ii) Testar muitos hiperparametros sem protocolo fixo pode induzir overfitting no conjunto de teste.'],
 '4_pipeline_confiavel': 'Parcialmente confiavel: ele e reprodutivel, modular e mede multiplas metricas, mas ainda precisa de validacao cruzada e analise de erro por classe para maior robustez.'}